# you basically do not need to run these codes, as all the processed data is stored in separate files in /results folder, for the stats however the files in /RE_DO folders are recommended

In [ ]:
import trompy as tp
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import matplotlib.patches as mpatches

# <font color="red">cell below was used as the main data processor generated most of the files used for stats/plots/paper</font>

In [1]:

import csv
from datetime import datetime
import math
import numpy as np
import trompy as tp

# ----------------------- Config -----------------------
METAFILE_PATH = "..\\src\\FEDProtein_METAFILE.xls"
DATA_FOLDER_FMT = "..\\data\\{}"     #
SHEET_NAME = "METAFILE"
EVENT_NAME = "Pellet"
LIGHTS_ON = "07:00:00"               # local lights-on anchor within each calendar day
MEAL_IPI_THRESHOLD_H = 1.0/60.0      # 60 seconds in hours
PHASE_DAYS = {"GRAIN": 3, "PR": 7, "NR": 7}
DEBUG_PRINTS = True                 # set True to print sanity warnings
# ------------------------------------------------------


# -------------------- Utility parsing --------------------
def _parse_timestamp(ts_str, formats):
    for fmt in formats:
        try:
            return datetime.strptime(ts_str, fmt), fmt
        except ValueError:
            continue
    raise ValueError(f"Cannot parse timestamp: {ts_str}")

def get_FEDevents(filename, eventname, lights_on=LIGHTS_ON):
    """
    Read a *single* day's CSV and return pellet times (in hours) referenced to that day's lights-on.
    Ensures times are sorted and de-duplicated (~1s tolerance).
    """
    formats = ['%Y-%m-%d %H:%M:%S', '%m/%d/%Y %H:%M:%S']
    rows = []

    try:
        with open(filename) as f:
            csvreader = csv.reader(f)
            next(csvreader)  # header
            for row in csvreader:
                rows.append(row)
    except FileNotFoundError:
        print(f"File not found: {filename}")
        return []

    if not rows:
        print(f"No data found in file: {filename}")
        return []

    # Detect date format from first row
    _, date_format = _parse_timestamp(rows[0][0], formats)

    # Reference point = same calendar day as first row + LIGHTS_ON (built with datetime.combine)
    first_dt = datetime.strptime(rows[0][0], date_format)
    lo_time = datetime.strptime(lights_on, "%H:%M:%S").time()
    refpoint = datetime.combine(first_dt.date(), lo_time)

    stamps = []
    for row in rows:
        if row[7] == eventname:
            dt = datetime.strptime(row[0], date_format)
            dh = (dt - refpoint).total_seconds() / 3600.0
            stamps.append(dh)

    if not stamps:
        return []

    # Sort and dedupe to ~1s tolerance
    stamps = np.array(sorted(stamps), dtype=float)
    if len(stamps) > 1:
        keep = [0]
        for i in range(1, len(stamps)):
            if (stamps[i] - stamps[keep[-1]]) > (1.0/3600.0):
                keep.append(i)
        stamps = stamps[keep]

    return stamps.tolist()



def _assign_day_hour(t, days=7):
    """
    Safe day/hour assignment for a continuous 0..days*24h timeline.
    Returns (day_index, hour) or (None, None) if out of range.
    """
    if t < 0:
        return None, None
    day = int(math.floor(t / 24.0))
    if day < 0 or day >= days:
        return None, None
    hour = int(math.floor(t)) % 24
    return day, hour
# ----------------------------------------------------------


# ------------------ Clustering & binning ------------------
def segment_events(pellettimes, meal_threshold=MEAL_IPI_THRESHOLD_H):
    """
    Split a continuous timestamp list/array into clusters using <= 60 s IPI rule.
    Accepts Python lists or NumPy arrays. Ignores NaNs. Returns list of clusters.
    """
    if pellettimes is None:
        return []
    ts = np.asarray(pellettimes, dtype=float).ravel()
    ts = ts[np.isfinite(ts)]
    if ts.size == 0:
        return []
    ts.sort()

    IPIs = np.diff(ts)
    clusters = []
    cur = [ts[0]]
    for i, ipi in enumerate(IPIs):
        if ipi <= meal_threshold:
            cur.append(ts[i + 1])
        else:
            clusters.append(cur)
            cur = [ts[i + 1]]
    clusters.append(cur)
    return clusters



def classify_and_bin(clusters, days=7):
    """
    Classify clusters into snacks(=1), meals(2–5), mega(>=6) and fill hourly/day bins
    using the cluster mid-time (stable at hour boundaries).
    Returns:
        snacks, meals, mega (lists of clusters),
        hourly_meals_per_day, hourly_snacks_per_day, hourly_mega_per_day (7x24 lists)
    """
    hourly_meals = [[0]*24 for _ in range(days)]
    hourly_snacks = [[0]*24 for _ in range(days)]
    hourly_mega   = [[0]*24 for _ in range(days)]

    snacks, meals, mega = [], [], []

    for ev in clusters:
        n = len(ev)
        t_mid = 0.5*(ev[0] + ev[-1])
        d, h = _assign_day_hour(t_mid, days=days)
        if n == 1:
            snacks.append(ev)
            if d is not None:
                hourly_snacks[d][h] += 1
        elif 2 <= n <= 5:
            meals.append(ev)
            if d is not None:
                hourly_meals[d][h] += 1
        elif n >= 6:
            mega.append(ev)
            if d is not None:
                hourly_mega[d][h] += 1

    return snacks, meals, mega, hourly_meals, hourly_snacks, hourly_mega


def rate_per_day_from_hourly(hourly_counts, hours_per_day=24):
    """
    Compute events/hour for each day using hourly bins.
    """
    freq = []
    for day_counts in hourly_counts:
        total_events = sum(day_counts)
        freq.append(total_events / float(hours_per_day))
    return freq


def avg_size_per_day_from_clusters(clusters, days=7):
    """
    Average cluster size per day (pellets/event), using cluster mid-time to assign day.
    """
    sums = [0]*days
    cnts = [0]*days
    for ev in clusters:
        d, _ = _assign_day_hour(0.5*(ev[0] + ev[-1]), days=days)
        if d is not None:
            sums[d] += len(ev)
            cnts[d] += 1
    sizes = [ (s/c) if c>0 else 0 for s,c in zip(sums,cnts) ]
    return sizes
# ----------------------------------------------------------


# ----------------- Phase-level computation ----------------
def get_meal_and_snack_metrics_week(pellettimes, days=7, meal_threshold=MEAL_IPI_THRESHOLD_H):
    """
    weekly metrics for a contiguous week timeline (0..days*24 hours).
    Returns:
      mealsize, snack_size, nmeals, meal_freq_phase, nsnacks, snack_freq_phase,
      mega_freq_phase, avg_mega_size,
      hourly_meals(7x24), hourly_snacks(7x24), hourly_mega(7x24),
      meals(list), snacks(list), mega(list), n_mega,
      meal_size_per_day(7), meal_freq_per_day(7),
      snack_size_per_day(7), snack_freq_per_day(7),
      mega_size_per_day(7), mega_freq_per_day(7)
    """
    if not pellettimes:
        zero_7_24 = [[0]*24 for _ in range(days)]
        zero7 = [0]*days
        return (0,0,0,0,0,0,0,0,
                zero_7_24, zero_7_24, zero_7_24,
                [], [], [], 0,
                zero7, zero7, zero7, zero7, zero7, zero7)

    # Keep only events in [0, days*24)
    TMAX = days * 24.0
    ts = np.array([t for t in pellettimes if (t >= 0.0 and t < TMAX)], dtype=float)
    if len(ts) == 0:
        zero_7_24 = [[0]*24 for _ in range(days)]
        zero7 = [0]*days
        return (0,0,0,0,0,0,0,0,
                zero_7_24, zero_7_24, zero_7_24,
                [], [], [], 0,
                zero7, zero7, zero7, zero7, zero7, zero7)

    ts.sort()

    clusters = segment_events(ts, meal_threshold=meal_threshold)
    snacks, meals, mega, H_meals, H_snacks, H_mega = classify_and_bin(clusters, days=days)

    nmeals = len(meals)
    nsnacks = len(snacks)
    nmega = len(mega)

    mealsize      = (sum(len(ev) for ev in meals) / nmeals) if nmeals else 0
    snack_size    = (sum(len(ev) for ev in snacks) / nsnacks) if nsnacks else 0
    avg_mega_size = (sum(len(ev) for ev in mega)  / nmega)  if nmega  else 0

    # Phase-level frequencies: normalised by full phase duration 
    total_hours = float(days * 24)
    meal_freq_phase = nmeals / total_hours
    snack_freq_phase = nsnacks / total_hours
    mega_freq_phase  = nmega  / total_hours

    # Per-day frequencies via hourly bins (events/day / 24)
    meal_freq_per_day  = rate_per_day_from_hourly(H_meals,  hours_per_day=24)
    snack_freq_per_day = rate_per_day_from_hourly(H_snacks, hours_per_day=24)
    mega_freq_per_day  = rate_per_day_from_hourly(H_mega,   hours_per_day=24)

    # Per-day sizes
    meal_size_per_day  = avg_size_per_day_from_clusters(meals, days=days)
    snack_size_per_day = avg_size_per_day_from_clusters(snacks, days=days)
    mega_size_per_day  = avg_size_per_day_from_clusters(mega,  days=days)

    return (mealsize, snack_size, nmeals, meal_freq_phase, nsnacks, snack_freq_phase,
            mega_freq_phase, avg_mega_size,
            H_meals, H_snacks, H_mega,
            meals, snacks, mega, nmega,
            meal_size_per_day, meal_freq_per_day,
            snack_size_per_day, snack_freq_per_day,
            mega_size_per_day, mega_freq_per_day)
# ----------------------------------------------------------


# -------------------- Phase loading w/ offsets --------------------
def load_phase_with_offsets(rows, mouse_id, phase, lights_on=LIGHTS_ON):
    """
    Concatenate all daily files for a phase, adding +24h offsets so the phase spans [0..days*24).
    Returns (phase_ts, daily_lists) where:
      - phase_ts: single list with offsets applied (continuous 0..days*24)
      - daily_lists: list of per-day local lists (each 0..24h relative to that day)
    """
    days = PHASE_DAYS.get(phase, 7)

    # Collect per-day files for this phase
    phase_files = [r for r in rows if (r[1] == mouse_id and r[3] == "FF" and r[2] == phase)]
    # If your filenames are sortable by date, you can uncomment the next line:
    # phase_files = sorted(phase_files, key=lambda r: r[0])

    phase_ts = []
    daily_lists = []
    for day_idx, r in enumerate(phase_files):
        filename = DATA_FOLDER_FMT.format(r[0])
        day_ts = get_FEDevents(filename, EVENT_NAME, lights_on=lights_on)  # 0..~24 within that day
        daily_lists.append(day_ts)
        offset = day_idx * 24.0
        phase_ts.extend([t + offset for t in day_ts])

    return phase_ts, daily_lists
# ------------------------------------------------------------------


# ------------------- Derived helper metrics -------------------
def get_pellets_per_day_contiguous(timestamps, days):
    """
    Count pellets per day on a contiguous 0..days*24 timeline.
    """
    pellets_per_day = [0] * days
    if not timestamps:
        return pellets_per_day
    ts = np.array(timestamps, dtype=float)
    for day in range(days):
        lo = day * 24.0
        hi = (day + 1) * 24.0
        pellets_per_day[day] = int(np.sum((ts >= lo) & (ts < hi)))
    return pellets_per_day

def _events_per_day_from_clusters(clusters, days):
    out = [0]*days
    for ev in clusters:
        tmid = 0.5*(ev[0]+ev[-1])
        d, _ = _assign_day_hour(tmid, days=days)
        if d is not None:
            out[d] += 1
    return out

def sanity_check_day_rates(label, freq_per_day, hard_cap=10.0):
    if not DEBUG_PRINTS:
        return
    for d, v in enumerate(freq_per_day):
        if v > hard_cap:
            print(f"[WARN] {label}: day {d} has {v:.2f} events/hour (suspicious).")
# --------------------------------------------------------------


# =========================== MAIN ===========================
# Load metafile
rows, header = tp.metafilereader(METAFILE_PATH, sheetname=SHEET_NAME)

# Build mice dict with sex/order
mice = {}
for r in rows:
    mouse_id = r[1]
    if mouse_id not in mice:
        mice[mouse_id] = {}
        mice[mouse_id]["sex"] = r[4]
        mice[mouse_id]["order"] = r[5]

# Process each mouse
for key in mice.keys():
    # Phase timestamps (continuous timelines)
    grain_ts, grain_daily = load_phase_with_offsets(rows, key, "GRAIN")
    pr_ts,    pr_daily    = load_phase_with_offsets(rows, key, "PR")
    nr_ts,    nr_daily    = load_phase_with_offsets(rows, key, "NR")

    mice[key]["grain_timestamps"] = grain_ts
    mice[key]["pr_timestamps"]    = pr_ts
    mice[key]["nr_timestamps"]    = nr_ts

    # weekly metrics
    # GRAIN (3 days)
    (mice[key]["grain_meal_size"],
     mice[key]["grain_snack_size"],
     mice[key]["grain_number_of_meals"],
     mice[key]["grain_meal_frequency"],
     mice[key]["grain_number_of_snacks"],
     mice[key]["grain_snack_frequency"],
     mice[key]["grain_mega_meal_frequency"],
     mice[key]["grain_mega_meal_size"],
     mice[key]["grain_hourly_meals"],
     mice[key]["grain_hourly_snacks"],
     mice[key]["grain_hourly_mega_meals"],
     grain_meals, grain_snacks, grain_mega, mice[key]["grain_number_of_mega_meals"],
     mice[key]["grain_meal_size_per_day"],  mice[key]["grain_meal_freq_per_day"],
     mice[key]["grain_snack_size_per_day"], mice[key]["grain_snack_freq_per_day"],
     mice[key]["grain_mega_meal_size_per_day"], mice[key]["grain_mega_meal_freq_per_day"]) = \
        get_meal_and_snack_metrics_week(grain_ts, days=PHASE_DAYS["GRAIN"])

    # PR (7 days)
    (mice[key]["pr_meal_size"],
     mice[key]["pr_snack_size"],
     mice[key]["pr_number_of_meals"],
     mice[key]["pr_meal_frequency"],
     mice[key]["pr_number_of_snacks"],
     mice[key]["pr_snack_frequency"],
     mice[key]["pr_mega_meal_frequency"],
     mice[key]["pr_mega_meal_size"],
     mice[key]["pr_hourly_meals"],
     mice[key]["pr_hourly_snacks"],
     mice[key]["pr_hourly_mega_meals"],
     pr_meals, pr_snacks, pr_mega, mice[key]["pr_number_of_mega_meals"],
     mice[key]["pr_meal_size_per_day"],  mice[key]["pr_meal_freq_per_day"],
     mice[key]["pr_snack_size_per_day"], mice[key]["pr_snack_freq_per_day"],
     mice[key]["pr_mega_meal_size_per_day"], mice[key]["pr_mega_meal_freq_per_day"]) = \
        get_meal_and_snack_metrics_week(pr_ts, days=PHASE_DAYS["PR"])

    # NR (7 days)
    (mice[key]["nr_meal_size"],
     mice[key]["nr_snack_size"],
     mice[key]["nr_number_of_meals"],
     mice[key]["nr_meal_frequency"],
     mice[key]["nr_number_of_snacks"],
     mice[key]["nr_snack_frequency"],
     mice[key]["nr_mega_meal_frequency"],
     mice[key]["nr_mega_meal_size"],
     mice[key]["nr_hourly_meals"],
     mice[key]["nr_hourly_snacks"],
     mice[key]["nr_hourly_mega_meals"],
     nr_meals, nr_snacks, nr_mega, mice[key]["nr_number_of_mega_meals"],
     mice[key]["nr_meal_size_per_day"],  mice[key]["nr_meal_freq_per_day"],
     mice[key]["nr_snack_size_per_day"], mice[key]["nr_snack_freq_per_day"],
     mice[key]["nr_mega_meal_size_per_day"], mice[key]["nr_mega_meal_freq_per_day"]) = \
        get_meal_and_snack_metrics_week(nr_ts, days=PHASE_DAYS["NR"])

    # ---------- Pellets/day on contiguous timelines ----------
    mice[key]["grain_pellets_per_day"] = get_pellets_per_day_contiguous(grain_ts, days=PHASE_DAYS["GRAIN"])
    mice[key]["pr_pellets_per_day"]    = get_pellets_per_day_contiguous(pr_ts,    days=PHASE_DAYS["PR"])
    mice[key]["nr_pellets_per_day"]    = get_pellets_per_day_contiguous(nr_ts,    days=PHASE_DAYS["NR"])

    # ---------- Events/day (sanity/alignment with earlier keys) ----------
    mice[key]["grain_meals_per_day"]      = _events_per_day_from_clusters(grain_meals, days=PHASE_DAYS["GRAIN"])
    mice[key]["grain_snacks_per_day"]     = _events_per_day_from_clusters(grain_snacks, days=PHASE_DAYS["GRAIN"])
    mice[key]["grain_mega_meals_per_day"] = _events_per_day_from_clusters(grain_mega,  days=PHASE_DAYS["GRAIN"])

    mice[key]["pr_meals_per_day"]      = _events_per_day_from_clusters(pr_meals, days=PHASE_DAYS["PR"])
    mice[key]["pr_snacks_per_day"]     = _events_per_day_from_clusters(pr_snacks, days=PHASE_DAYS["PR"])
    mice[key]["pr_mega_meals_per_day"] = _events_per_day_from_clusters(pr_mega,  days=PHASE_DAYS["PR"])

    mice[key]["nr_meals_per_day"]      = _events_per_day_from_clusters(nr_meals, days=PHASE_DAYS["NR"])
    mice[key]["nr_snacks_per_day"]     = _events_per_day_from_clusters(nr_snacks, days=PHASE_DAYS["NR"])
    mice[key]["nr_mega_meals_per_day"] = _events_per_day_from_clusters(nr_mega,  days=PHASE_DAYS["NR"])

    # ---------- Combined full 17-day vectors (3 + 7 + 7) ----------
    mice[key]["all_pellets_per_day"] = (
        mice[key].get("grain_pellets_per_day", [0]*PHASE_DAYS["GRAIN"]) +
        mice[key].get("pr_pellets_per_day",    [0]*PHASE_DAYS["PR"]) +
        mice[key].get("nr_pellets_per_day",    [0]*PHASE_DAYS["NR"])
    )
    mice[key]["all_meals_per_day"] = (
        mice[key].get("grain_meals_per_day", [0]*PHASE_DAYS["GRAIN"]) +
        mice[key].get("pr_meals_per_day",    [0]*PHASE_DAYS["PR"]) +
        mice[key].get("nr_meals_per_day",    [0]*PHASE_DAYS["NR"])
    )
    mice[key]["all_snacks_per_day"] = (
        mice[key].get("grain_snacks_per_day", [0]*PHASE_DAYS["GRAIN"]) +
        mice[key].get("pr_snacks_per_day",    [0]*PHASE_DAYS["PR"]) +
        mice[key].get("nr_snacks_per_day",    [0]*PHASE_DAYS["NR"])
    )
    mice[key]["all_mega_meals_per_day"] = (
        mice[key].get("grain_mega_meals_per_day", [0]*PHASE_DAYS["GRAIN"]) +
        mice[key].get("pr_mega_meals_per_day",    [0]*PHASE_DAYS["PR"]) +
        mice[key].get("nr_mega_meals_per_day",    [0]*PHASE_DAYS["NR"])
    )

    # ---------- Sanity checks (optional) ----------
    sanity_check_day_rates("GRAIN snack freq", mice[key]["grain_snack_freq_per_day"])
    sanity_check_day_rates("PR snack freq",    mice[key]["pr_snack_freq_per_day"])
    sanity_check_day_rates("NR snack freq",    mice[key]["nr_snack_freq_per_day"])

    sanity_check_day_rates("GRAIN mega freq",  mice[key]["grain_mega_meal_freq_per_day"])
    sanity_check_day_rates("PR mega freq",     mice[key]["pr_mega_meal_freq_per_day"])
    sanity_check_day_rates("NR mega freq",     mice[key]["nr_mega_meal_freq_per_day"])


if DEBUG_PRINTS:
    for key in mice:
        print(f"\nMouse {key} | Sex={mice[key]['sex']} | Order={mice[key]['order']}")
        print("  PR snack freq/day:",  np.round(mice[key]["pr_snack_freq_per_day"], 3))
        print("  NR snack freq/day:",  np.round(mice[key]["nr_snack_freq_per_day"], 3))
        print("  PR mega  freq/day:",  np.round(mice[key]["pr_mega_meal_freq_per_day"], 3))
        print("  NR mega  freq/day:",  np.round(mice[key]["nr_mega_meal_freq_per_day"], 3))
        print("  PR hourly mega map (day0):", mice[key]["pr_hourly_mega_meals"][0])
        print("  NR hourly mega map (day0):", mice[key]["nr_hourly_mega_meals"][0])




Mouse FEDXA01 | Sex=M | Order=2.0
  PR snack freq/day: [1.333 1.958 2.208 2.583 1.208 1.083 0.542]
  NR snack freq/day: [0.542 0.583 0.583 0.792 0.75  0.5   0.542]
  PR mega  freq/day: [0.167 0.042 0.125 0.25  0.542 0.667 0.708]
  NR mega  freq/day: [0.875 0.75  0.708 0.667 0.75  0.833 0.833]
  PR hourly mega map (day0): [0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  NR hourly mega map (day0): [0, 0, 0, 2, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 2, 3, 2, 1, 0, 0, 1, 1, 1]

Mouse FEDXA02 | Sex=M | Order=2.0
  PR snack freq/day: [0.458 1.042 1.375 1.375 0.917 0.667 0.708]
  NR snack freq/day: [0.542 0.708 0.667 0.542 0.667 0.5   0.5  ]
  PR mega  freq/day: [0.167 0.042 0.083 0.083 0.167 0.208 0.292]
  NR mega  freq/day: [0.708 0.458 0.25  0.375 0.292 0.25  0.542]
  PR hourly mega map (day0): [0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]
  NR hourly mega map (day0): [0, 0, 0, 1, 2, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 3, 0, 2, 0, 1, 1, 0, 0]

M

# code below just saves the results of the data processing from the previous cell  

In [ ]:
# so jaime asked to have the per-day data of feeding metrics too 

import os
import re
import pandas as pd

# ---------- Config ----------
out_dir = "../results/FIVE/Fixed_Frequency_dynamics" # my old repo code, change as needed
wide_csv  = os.path.join(out_dir, "Metrics_per_day_freq_fixed_wide.csv")
long_csv  = os.path.join(out_dir, "Metrics_per_day_freq_fixed_long.csv")
save_param_phase_splits = True  # set False if you don't want the extra split files
# ----------------------------

os.makedirs(out_dir, exist_ok=True)

# Phases and daily parameters we expect in `mice`
phases = ["grain", "pr", "nr"]
daily_params = [
    "meals_per_day", "snacks_per_day", "mega_meals_per_day",
    "meal_size_per_day", "meal_freq_per_day",
    "snack_size_per_day", "snack_freq_per_day",
    "mega_meal_size_per_day", "mega_meal_freq_per_day",
    "pellets_per_day"
]

# ---------- Build WIDE table ----------
rows = []
for mouse_id, mouse_data in mice.items():
    row = {
        "mouse_id": mouse_id,
        "sex": mouse_data.get("sex"),
        "order": mouse_data.get("order"),
    }

    for phase in phases:
        for param in daily_params:
            key = f"{phase}_{param}"
            values = mouse_data.get(key, [])
            # Write per-day columns; 1-based day index in headings
            for i, val in enumerate(values):
                row[f"{phase}_{param}_day{i+1}"] = val

    rows.append(row)

df_wide = pd.DataFrame(rows)

# Put id/sex/order first
base_cols = ["mouse_id", "sex", "order"]
other_cols = [c for c in df_wide.columns if c not in base_cols]
df_wide = df_wide[base_cols + sorted(other_cols)]

# Save wide
df_wide.to_csv(wide_csv, index=False)
print(f"[OK] WIDE table saved -> {wide_csv}")


# long rows: mouse_id, sex, order, phase, param, day (1-based), value
long_records = []
for _, r in df_wide.iterrows():
    for phase in phases:
        for param in daily_params:
            # Collect all columns matching e.g. "pr_meal_freq_per_day_day(\d+)"
            prefix = f"{phase}_{param}_day"
            matching = [c for c in df_wide.columns if c.startswith(prefix)]
            if not matching:
                continue
            for c in matching:
                # extract day number from column name
                m = re.search(r"_day(\d+)$", c)
                if not m:
                    continue
                day = int(m.group(1))
                long_records.append({
                    "mouse_id": r["mouse_id"],
                    "sex": r["sex"],
                    "order": r["order"],
                    "phase": phase.upper(),       # GRAIN/PR/NR in caps if you prefer
                    "param": param,               # keep param name as in keys
                    "day": day,                   # 1-based day within phase
                    "value": r[c]
                })

df_long = pd.DataFrame(long_records)
# Optional: sort for readability
df_long = df_long.sort_values(["param", "phase", "mouse_id", "day"]).reset_index(drop=True)

# Save long
df_long.to_csv(long_csv, index=False)
print(f"[OK] LONG (tidy) table saved -> {long_csv}")


if save_param_phase_splits:
    for phase in phases:
        for param in daily_params:
            prefix = f"{phase}_{param}_day"
            cols = [c for c in df_wide.columns if c.startswith(prefix)]
            if not cols:
                continue
            out_name = f"{phase.upper()}_{param}.csv"
            out_path = os.path.join(out_dir, out_name)
            sub = df_wide[["mouse_id", "sex", "order"] + sorted(cols)].copy()
            sub.to_csv(out_path, index=False)
            print(f"[OK] Split saved -> {out_path}")


# codes above was the old code I used for processing of raw data that were used to do the stats/plots
### if needed we can also use the cleaner code below


In [ ]:
import csv
import math
import os
import re
from datetime import datetime

import numpy as np
import pandas as pd
import trompy as tp

# =========================================================
# CONFIG
# =========================================================
METAFILE_PATH = r"..\FEDProtein_METAFILE.xls"
DATA_FOLDER_FMT = r"..\data\{}"
SHEET_NAME = "METAFILE"
EVENT_NAME = "Pellet"
LIGHTS_ON = "07:00:00"
MEAL_IPI_THRESHOLD_H = 1.0 / 60.0   # 60 sec in hours
PHASE_DAYS = {"GRAIN": 3, "PR": 7, "NR": 7}
DEBUG_PRINTS = True


DEDUP_WITHIN_1S = True

OUT_DIR = r"..\results\FIVE\CORRECTED_REALIGNED_FOR_PLOTTING"
os.makedirs(OUT_DIR, exist_ok=True)

# =========================================================
# HELPERS
# =========================================================
def _parse_timestamp(ts_str, formats):
    for fmt in formats:
        try:
            return datetime.strptime(ts_str, fmt), fmt
        except ValueError:
            continue
    raise ValueError(f"Cannot parse timestamp: {ts_str}")

def get_FEDevents(filename, eventname, lights_on=LIGHTS_ON, dedup_within_1s=DEDUP_WITHIN_1S):
    formats = ['%Y-%m-%d %H:%M:%S', '%m/%d/%Y %H:%M:%S']
    rows = []

    try:
        with open(filename, newline='') as f:
            csvreader = csv.reader(f)
            next(csvreader, None)
            for row in csvreader:
                rows.append(row)
    except FileNotFoundError:
        print(f"File not found: {filename}")
        return []

    if not rows:
        print(f"No data found in file: {filename}")
        return []

    _, date_format = _parse_timestamp(rows[0][0], formats)

    first_dt = datetime.strptime(rows[0][0], date_format)
    lo_time = datetime.strptime(lights_on, "%H:%M:%S").time()
    refpoint = datetime.combine(first_dt.date(), lo_time)

    stamps = []
    for row in rows:
        if len(row) > 7 and row[7] == eventname:
            dt = datetime.strptime(row[0], date_format)
            dh = (dt - refpoint).total_seconds() / 3600.0
            stamps.append(dh)

    if not stamps:
        return []

    stamps = np.array(sorted(stamps), dtype=float)

    if dedup_within_1s and len(stamps) > 1:
        keep = [0]
        for i in range(1, len(stamps)):
            if (stamps[i] - stamps[keep[-1]]) > (1.0 / 3600.0):
                keep.append(i)
        stamps = stamps[keep]

    return stamps.tolist()

def _assign_day_hour(t, days=7):
    if t < 0:
        return None, None
    day = int(math.floor(t / 24.0))
    if day < 0 or day >= days:
        return None, None
    hour = int(math.floor(t)) % 24
    return day, hour

def segment_events(pellettimes, meal_threshold=MEAL_IPI_THRESHOLD_H):
    if pellettimes is None:
        return []

    ts = np.asarray(pellettimes, dtype=float).ravel()
    ts = ts[np.isfinite(ts)]
    if ts.size == 0:
        return []

    ts.sort()
    ipis = np.diff(ts)

    clusters = []
    cur = [ts[0]]
    for i, ipi in enumerate(ipis):
        if ipi <= meal_threshold:
            cur.append(ts[i + 1])
        else:
            clusters.append(cur)
            cur = [ts[i + 1]]
    clusters.append(cur)
    return clusters

def classify_and_bin(clusters, days=7):
    hourly_meals = [[0] * 24 for _ in range(days)]
    hourly_snacks = [[0] * 24 for _ in range(days)]
    hourly_mega = [[0] * 24 for _ in range(days)]

    snacks, meals, mega = [], [], []

    for ev in clusters:
        n = len(ev)
        t_mid = 0.5 * (ev[0] + ev[-1])
        d, h = _assign_day_hour(t_mid, days=days)

        if n == 1:
            snacks.append(ev)
            if d is not None:
                hourly_snacks[d][h] += 1
        elif 2 <= n <= 5:
            meals.append(ev)
            if d is not None:
                hourly_meals[d][h] += 1
        elif n >= 6:
            mega.append(ev)
            if d is not None:
                hourly_mega[d][h] += 1

    return snacks, meals, mega, hourly_meals, hourly_snacks, hourly_mega

def rate_per_day_from_hourly(hourly_counts, hours_per_day=24):
    return [sum(day_counts) / float(hours_per_day) for day_counts in hourly_counts]

def avg_size_per_day_from_clusters(clusters, days=7):
    sums = [0] * days
    cnts = [0] * days
    for ev in clusters:
        d, _ = _assign_day_hour(0.5 * (ev[0] + ev[-1]), days=days)
        if d is not None:
            sums[d] += len(ev)
            cnts[d] += 1
    return [(s / c) if c > 0 else 0 for s, c in zip(sums, cnts)]

def get_meal_and_snack_metrics_week(pellettimes, days=7, meal_threshold=MEAL_IPI_THRESHOLD_H):
    zero_7_24 = [[0] * 24 for _ in range(days)]
    zero7 = [0] * days

    if not pellettimes:
        return (
            0, 0, 0, 0, 0, 0, 0, 0,
            zero_7_24, zero_7_24, zero_7_24,
            [], [], [], 0,
            zero7, zero7, zero7, zero7, zero7, zero7
        )

    tmax = days * 24.0
    ts = np.array([t for t in pellettimes if (0.0 <= t < tmax)], dtype=float)

    if len(ts) == 0:
        return (
            0, 0, 0, 0, 0, 0, 0, 0,
            zero_7_24, zero_7_24, zero_7_24,
            [], [], [], 0,
            zero7, zero7, zero7, zero7, zero7, zero7
        )

    ts.sort()

    clusters = segment_events(ts, meal_threshold=meal_threshold)
    snacks, meals, mega, H_meals, H_snacks, H_mega = classify_and_bin(clusters, days=days)

    nmeals = len(meals)
    nsnacks = len(snacks)
    nmega = len(mega)

    mealsize = (sum(len(ev) for ev in meals) / nmeals) if nmeals else 0
    snack_size = (sum(len(ev) for ev in snacks) / nsnacks) if nsnacks else 0
    avg_mega_size = (sum(len(ev) for ev in mega) / nmega) if nmega else 0

    total_hours = float(days * 24)
    meal_freq_phase = nmeals / total_hours
    snack_freq_phase = nsnacks / total_hours
    mega_freq_phase = nmega / total_hours

    meal_freq_per_day = rate_per_day_from_hourly(H_meals, hours_per_day=24)
    snack_freq_per_day = rate_per_day_from_hourly(H_snacks, hours_per_day=24)
    mega_freq_per_day = rate_per_day_from_hourly(H_mega, hours_per_day=24)

    meal_size_per_day = avg_size_per_day_from_clusters(meals, days=days)
    snack_size_per_day = avg_size_per_day_from_clusters(snacks, days=days)
    mega_size_per_day = avg_size_per_day_from_clusters(mega, days=days)

    return (
        mealsize, snack_size, nmeals, meal_freq_phase, nsnacks, snack_freq_phase,
        mega_freq_phase, avg_mega_size,
        H_meals, H_snacks, H_mega,
        meals, snacks, mega, nmega,
        meal_size_per_day, meal_freq_per_day,
        snack_size_per_day, snack_freq_per_day,
        mega_size_per_day, mega_freq_per_day
    )

def load_phase_with_offsets(rows, mouse_id, phase, lights_on=LIGHTS_ON):
    phase_files = [r for r in rows if (r[1] == mouse_id and r[3] == "FF" and r[2] == phase)]

    # Keep as recorded order from metafile unless you explicitly want to sort:
    # phase_files = sorted(phase_files, key=lambda r: r[0])

    phase_ts = []
    daily_lists = []

    for day_idx, r in enumerate(phase_files):
        filename = DATA_FOLDER_FMT.format(r[0])
        day_ts = get_FEDevents(filename, EVENT_NAME, lights_on=lights_on, dedup_within_1s=DEDUP_WITHIN_1S)
        daily_lists.append(day_ts)
        offset = day_idx * 24.0
        phase_ts.extend([t + offset for t in day_ts])

    return phase_ts, daily_lists

def get_pellets_per_day_contiguous(timestamps, days):
    pellets_per_day = [0] * days
    if not timestamps:
        return pellets_per_day

    ts = np.array(timestamps, dtype=float)
    for day in range(days):
        lo = day * 24.0
        hi = (day + 1) * 24.0
        pellets_per_day[day] = int(np.sum((ts >= lo) & (ts < hi)))
    return pellets_per_day

def _events_per_day_from_clusters(clusters, days):
    out = [0] * days
    for ev in clusters:
        t_mid = 0.5 * (ev[0] + ev[-1])
        d, _ = _assign_day_hour(t_mid, days=days)
        if d is not None:
            out[d] += 1
    return out

def sanity_check_day_rates(label, freq_per_day, hard_cap=10.0):
    if not DEBUG_PRINTS:
        return
    for d, v in enumerate(freq_per_day):
        if v > hard_cap:
            print(f"[WARN] {label}: day {d} has {v:.2f} events/hour (suspicious).")

# =========================================================
# BUILD MICE DICTIONARY
# =========================================================
rows, header = tp.metafilereader(METAFILE_PATH, sheetname=SHEET_NAME)

mice = {}
for r in rows:
    mouse_id = r[1]
    if mouse_id not in mice:
        mice[mouse_id] = {}
        mice[mouse_id]["sex"] = r[4]
        mice[mouse_id]["order"] = r[5]

for key in mice.keys():
    grain_ts, _ = load_phase_with_offsets(rows, key, "GRAIN")
    pr_ts, _ = load_phase_with_offsets(rows, key, "PR")
    nr_ts, _ = load_phase_with_offsets(rows, key, "NR")

    mice[key]["grain_timestamps"] = grain_ts
    mice[key]["pr_timestamps"] = pr_ts
    mice[key]["nr_timestamps"] = nr_ts

    (
        mice[key]["grain_meal_size"],
        mice[key]["grain_snack_size"],
        mice[key]["grain_number_of_meals"],
        mice[key]["grain_meal_frequency"],
        mice[key]["grain_number_of_snacks"],
        mice[key]["grain_snack_frequency"],
        mice[key]["grain_mega_meal_frequency"],
        mice[key]["grain_mega_meal_size"],
        mice[key]["grain_hourly_meals"],
        mice[key]["grain_hourly_snacks"],
        mice[key]["grain_hourly_mega_meals"],
        grain_meals, grain_snacks, grain_mega, mice[key]["grain_number_of_mega_meals"],
        mice[key]["grain_meal_size_per_day"], mice[key]["grain_meal_freq_per_day"],
        mice[key]["grain_snack_size_per_day"], mice[key]["grain_snack_freq_per_day"],
        mice[key]["grain_mega_meal_size_per_day"], mice[key]["grain_mega_meal_freq_per_day"]
    ) = get_meal_and_snack_metrics_week(grain_ts, days=PHASE_DAYS["GRAIN"])

    (
        mice[key]["pr_meal_size"],
        mice[key]["pr_snack_size"],
        mice[key]["pr_number_of_meals"],
        mice[key]["pr_meal_frequency"],
        mice[key]["pr_number_of_snacks"],
        mice[key]["pr_snack_frequency"],
        mice[key]["pr_mega_meal_frequency"],
        mice[key]["pr_mega_meal_size"],
        mice[key]["pr_hourly_meals"],
        mice[key]["pr_hourly_snacks"],
        mice[key]["pr_hourly_mega_meals"],
        pr_meals, pr_snacks, pr_mega, mice[key]["pr_number_of_mega_meals"],
        mice[key]["pr_meal_size_per_day"], mice[key]["pr_meal_freq_per_day"],
        mice[key]["pr_snack_size_per_day"], mice[key]["pr_snack_freq_per_day"],
        mice[key]["pr_mega_meal_size_per_day"], mice[key]["pr_mega_meal_freq_per_day"]
    ) = get_meal_and_snack_metrics_week(pr_ts, days=PHASE_DAYS["PR"])

    (
        mice[key]["nr_meal_size"],
        mice[key]["nr_snack_size"],
        mice[key]["nr_number_of_meals"],
        mice[key]["nr_meal_frequency"],
        mice[key]["nr_number_of_snacks"],
        mice[key]["nr_snack_frequency"],
        mice[key]["nr_mega_meal_frequency"],
        mice[key]["nr_mega_meal_size"],
        mice[key]["nr_hourly_meals"],
        mice[key]["nr_hourly_snacks"],
        mice[key]["nr_hourly_mega_meals"],
        nr_meals, nr_snacks, nr_mega, mice[key]["nr_number_of_mega_meals"],
        mice[key]["nr_meal_size_per_day"], mice[key]["nr_meal_freq_per_day"],
        mice[key]["nr_snack_size_per_day"], mice[key]["nr_snack_freq_per_day"],
        mice[key]["nr_mega_meal_size_per_day"], mice[key]["nr_mega_meal_freq_per_day"]
    ) = get_meal_and_snack_metrics_week(nr_ts, days=PHASE_DAYS["NR"])

    mice[key]["grain_pellets_per_day"] = get_pellets_per_day_contiguous(grain_ts, days=PHASE_DAYS["GRAIN"])
    mice[key]["pr_pellets_per_day"] = get_pellets_per_day_contiguous(pr_ts, days=PHASE_DAYS["PR"])
    mice[key]["nr_pellets_per_day"] = get_pellets_per_day_contiguous(nr_ts, days=PHASE_DAYS["NR"])

    mice[key]["grain_meals_per_day"] = _events_per_day_from_clusters(grain_meals, days=PHASE_DAYS["GRAIN"])
    mice[key]["grain_snacks_per_day"] = _events_per_day_from_clusters(grain_snacks, days=PHASE_DAYS["GRAIN"])
    mice[key]["grain_mega_meals_per_day"] = _events_per_day_from_clusters(grain_mega, days=PHASE_DAYS["GRAIN"])

    mice[key]["pr_meals_per_day"] = _events_per_day_from_clusters(pr_meals, days=PHASE_DAYS["PR"])
    mice[key]["pr_snacks_per_day"] = _events_per_day_from_clusters(pr_snacks, days=PHASE_DAYS["PR"])
    mice[key]["pr_mega_meals_per_day"] = _events_per_day_from_clusters(pr_mega, days=PHASE_DAYS["PR"])

    mice[key]["nr_meals_per_day"] = _events_per_day_from_clusters(nr_meals, days=PHASE_DAYS["NR"])
    mice[key]["nr_snacks_per_day"] = _events_per_day_from_clusters(nr_snacks, days=PHASE_DAYS["NR"])
    mice[key]["nr_mega_meals_per_day"] = _events_per_day_from_clusters(nr_mega, days=PHASE_DAYS["NR"])

    sanity_check_day_rates("GRAIN snack freq", mice[key]["grain_snack_freq_per_day"])
    sanity_check_day_rates("PR snack freq", mice[key]["pr_snack_freq_per_day"])
    sanity_check_day_rates("NR snack freq", mice[key]["nr_snack_freq_per_day"])
    sanity_check_day_rates("GRAIN mega freq", mice[key]["grain_mega_meal_freq_per_day"])
    sanity_check_day_rates("PR mega freq", mice[key]["pr_mega_meal_freq_per_day"])
    sanity_check_day_rates("NR mega freq", mice[key]["nr_mega_meal_freq_per_day"])

# =========================================================
# EXPORT HELPERS
# =========================================================
def _safe_vals(mouse_data, phase, key, n):
    vals = mouse_data.get(f"{phase}_{key}", [])
    vals = list(vals) if vals is not None else []
    if len(vals) < n:
        vals = vals + [0] * (n - len(vals))
    return vals[:n]

def _base_row(mouse_id, mouse_data):
    return {
        "Mouse": mouse_id,
        "Sex": mouse_data.get("sex"),
        "Order": mouse_data.get("order"),
    }

def build_trend_file(mice, metric_key):
    rows_out = []
    for mouse_id, mouse_data in mice.items():
        row = _base_row(mouse_id, mouse_data)

        g = _safe_vals(mouse_data, "grain", metric_key, 3)
        nr = _safe_vals(mouse_data, "nr", metric_key, 7)
        pr = _safe_vals(mouse_data, "pr", metric_key, 7)

        row["G0"], row["G1"], row["G2"] = g
        for i in range(7):
            row[f"NR{i}"] = nr[i]
        for i in range(7):
            row[f"PR{i}"] = pr[i]

        rows_out.append(row)

    cols = ["Mouse", "Sex", "Order", "G0", "G1", "G2"] + \
           [f"NR{i}" for i in range(7)] + \
           [f"PR{i}" for i in range(7)]
    return pd.DataFrame(rows_out)[cols]

def build_realigned_final_file(mice, metric_key):
    rows_out = []
    for mouse_id, mouse_data in mice.items():
        row = _base_row(mouse_id, mouse_data)

        g = _safe_vals(mouse_data, "grain", metric_key, 3)
        pr = _safe_vals(mouse_data, "pr", metric_key, 7)
        nr = _safe_vals(mouse_data, "nr", metric_key, 7)

        row["G0"], row["G1"], row["G2"] = g
        for i in range(7):
            row[f"PR{i}"] = pr[i]
        for i in range(7):
            row[f"NR{i}"] = nr[i]

        rows_out.append(row)

    cols = ["Mouse", "Sex", "Order", "G0", "G1", "G2"] + \
           [f"PR{i}" for i in range(7)] + \
           [f"NR{i}" for i in range(7)]
    return pd.DataFrame(rows_out)[cols]

def build_master_wide(mice):
    phases = ["grain", "pr", "nr"]
    daily_params = [
        "meals_per_day", "snacks_per_day", "mega_meals_per_day",
        "meal_size_per_day", "meal_freq_per_day",
        "snack_size_per_day", "snack_freq_per_day",
        "mega_meal_size_per_day", "mega_meal_freq_per_day",
        "pellets_per_day"
    ]

    rows_out = []
    for mouse_id, mouse_data in mice.items():
        row = {
            "mouse_id": mouse_id,
            "sex": mouse_data.get("sex"),
            "order": mouse_data.get("order"),
        }
        for phase in phases:
            for param in daily_params:
                key = f"{phase}_{param}"
                values = mouse_data.get(key, [])
                for i, val in enumerate(values):
                    row[f"{phase}_{param}_day{i+1}"] = val
        rows_out.append(row)

    df = pd.DataFrame(rows_out)
    base_cols = ["mouse_id", "sex", "order"]
    other_cols = [c for c in df.columns if c not in base_cols]
    return df[base_cols + sorted(other_cols)]

def build_master_long(df_wide):
    phases = ["grain", "pr", "nr"]
    daily_params = [
        "meals_per_day", "snacks_per_day", "mega_meals_per_day",
        "meal_size_per_day", "meal_freq_per_day",
        "snack_size_per_day", "snack_freq_per_day",
        "mega_meal_size_per_day", "mega_meal_freq_per_day",
        "pellets_per_day"
    ]

    long_records = []
    for _, r in df_wide.iterrows():
        for phase in phases:
            for param in daily_params:
                prefix = f"{phase}_{param}_day"
                matching = [c for c in df_wide.columns if c.startswith(prefix)]
                for c in matching:
                    m = re.search(r"_day(\d+)$", c)
                    if not m:
                        continue
                    long_records.append({
                        "mouse_id": r["mouse_id"],
                        "sex": r["sex"],
                        "order": r["order"],
                        "phase": phase.upper(),
                        "param": param,
                        "day": int(m.group(1)),
                        "value": r[c]
                    })

    df_long = pd.DataFrame(long_records)
    return df_long.sort_values(["param", "phase", "mouse_id", "day"]).reset_index(drop=True)

def build_all_values_file(mice):
    metric_blocks = [
        "pellets_per_day",
        "meals_per_day",
        "snacks_per_day",
        "mega_meals_per_day",
        "meal_size_per_day",
        "meal_freq_per_day",
        "snack_size_per_day",
        "snack_freq_per_day",
        "mega_meal_size_per_day",
        "mega_meal_freq_per_day",
    ]

    rows_out = []
    for mouse_id, mouse_data in mice.items():
        row = _base_row(mouse_id, mouse_data)

        for metric_key in metric_blocks:
            g = _safe_vals(mouse_data, "grain", metric_key, 3)
            nr = _safe_vals(mouse_data, "nr", metric_key, 7)
            pr = _safe_vals(mouse_data, "pr", metric_key, 7)

            row[f"{metric_key}_G0"] = g[0]
            row[f"{metric_key}_G1"] = g[1]
            row[f"{metric_key}_G2"] = g[2]

            for i in range(7):
                row[f"{metric_key}_NR{i}"] = nr[i]
            for i in range(7):
                row[f"{metric_key}_PR{i}"] = pr[i]

        rows_out.append(row)

    return pd.DataFrame(rows_out)

# =========================================================
# EXPORT FILES
# =========================================================
build_trend_file(mice, "pellets_per_day").to_csv(
    os.path.join(OUT_DIR, "Pellets_trend.csv"), index=False
)
build_trend_file(mice, "meals_per_day").to_csv(
    os.path.join(OUT_DIR, "meals_per_day_trend.csv"), index=False
)
build_trend_file(mice, "snacks_per_day").to_csv(
    os.path.join(OUT_DIR, "snacks_per_day_trend.csv"), index=False
)
build_trend_file(mice, "mega_meals_per_day").to_csv(
    os.path.join(OUT_DIR, "mega_meals_per_day_trend.csv"), index=False
)

build_realigned_final_file(mice, "meal_size_per_day").to_csv(
    os.path.join(OUT_DIR, "meal_size_realigned_FINAL.csv"), index=False
)
build_realigned_final_file(mice, "meal_freq_per_day").to_csv(
    os.path.join(OUT_DIR, "meal_freq_realigned_FINAL.csv"), index=False
)
build_realigned_final_file(mice, "snack_size_per_day").to_csv(
    os.path.join(OUT_DIR, "snack_size_realigned_FINAL.csv"), index=False
)
build_realigned_final_file(mice, "snack_freq_per_day").to_csv(
    os.path.join(OUT_DIR, "snack_freq_realigned_FINAL.csv"), index=False
)
build_realigned_final_file(mice, "mega_meal_size_per_day").to_csv(
    os.path.join(OUT_DIR, "mega_meal_size_realigned_FINAL.csv"), index=False
)
build_realigned_final_file(mice, "mega_meal_freq_per_day").to_csv(
    os.path.join(OUT_DIR, "mega_meal_freq_realigned_FINAL.csv"), index=False
)

df_wide = build_master_wide(mice)
df_wide.to_csv(os.path.join(OUT_DIR, "Metrics_per_day_freq_fixed_wide.csv"), index=False)

df_long = build_master_long(df_wide)
df_long.to_csv(os.path.join(OUT_DIR, "Metrics_per_day_freq_fixed_long.csv"), index=False)

build_all_values_file(mice).to_csv(
    os.path.join(OUT_DIR, "ALL_CORRECTED_VALUES_together.csv"), index=False
)

print(f"[OK] Files saved to: {OUT_DIR}")